# MMAudio Fine-tuned 模型评估（MacBook M1 Pro）

这个 notebook 面向你当前仓库的 fine-tune 流程（`finetune_mmaudio_m1.py`），目标是：

1. 在 **MPS/CPU** 上加载 fine-tune 后的 checkpoint。
2. 做一组 **定性评估**（生成音频并保存）。
3. 做一组 **定量评估**（在提取好的 memmap latent 上计算 flow matching loss）。

> 推荐在仓库根目录启动 Jupyter，并使用与你训练一致的 Python 环境。

In [ ]:
# ===== 0) 环境检查 =====
import os
import json
from pathlib import Path
from dataclasses import dataclass

import torch
import pandas as pd
import tensordict as td
import torchaudio
from tqdm.auto import tqdm

print(f"Torch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"CUDA available: {torch.cuda.is_available()}")

def pick_device(prefer='mps'):
    if prefer == 'mps' and torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')

device = pick_device('mps')
# MPS 下建议 float32，稳定优先
dtype = torch.float32
print('Using device:', device, '| dtype:', dtype)

In [ ]:
# ===== 1) 导入 MMAudio 组件（对齐仓库脚本） =====
from mmaudio.eval_utils import all_model_cfg, generate, load_video
from mmaudio.model.networks import get_my_mmaudio
from mmaudio.model.flow_matching import FlowMatching
from mmaudio.model.utils.features_utils import FeaturesUtils

# 你在 finetune_mmaudio_m1.py 中常用 small_16k，这里默认也用它
MODEL_VARIANT = 'small_16k'  # 可改：small_44k / medium_44k / large_44k
BASE_CFG = all_model_cfg[MODEL_VARIANT]
print(BASE_CFG)

In [ ]:
# ===== 2) 评估参数配置 =====
# 你的 fine-tune 输出通常在: output/finetune/<exp_id>/checkpoints
EXP_ID = 'finetune_m1'  # 改成你的实验名
CHECKPOINT_PATH = Path(f'./output/finetune/{EXP_ID}/checkpoints/checkpoint_best.pth')
# 也支持 model_best.pth（纯 state_dict）

# 定性评估输入
VIDEO_TSV = Path('./training/my_video_train.tsv')
VIDEO_ROOT = Path('./training_videos')

# 定量评估输入（和 finetune_mmaudio_m1.py 对齐）
MEMMAP_DIR = Path('./output/memmap/vgg-my_videos')
MEMMAP_TSV = Path('./output/memmap/vgg-my_videos.tsv')

# 输出
OUTPUT_DIR = Path(f'./output/eval_m1/{EXP_ID}')
AUDIO_OUT_DIR = OUTPUT_DIR / 'audio_samples'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_OUT_DIR.mkdir(parents=True, exist_ok=True)

# 评估超参
DURATION_SEC = 8.0
NUM_STEPS = 25
CFG_STRENGTH = 4.5
SEED = 42

# 跑几个样本（M1 上建议先小规模）
NUM_QUALITATIVE = 5
NUM_QUANT_BATCHES = 20
BATCH_SIZE = 2

print('CHECKPOINT_PATH:', CHECKPOINT_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
# ===== 3) 加载 base 模型 + 覆盖 fine-tuned 权重 =====
def load_finetuned_net(model_variant: str, checkpoint_path: Path, device: torch.device, dtype=torch.float32):
    net = get_my_mmaudio(model_variant).to(device=device, dtype=dtype).eval()

    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

    ckpt = torch.load(checkpoint_path, map_location='cpu')

    # 兼容两类保存格式：
    # 1) {'model': state_dict, ...} -> finetune_mmaudio_m1.py checkpoint_xxx.pth
    # 2) 纯 state_dict -> model_best.pth
    state_dict = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt

    missing, unexpected = net.load_state_dict(state_dict, strict=False)
    print('Loaded checkpoint:', checkpoint_path)
    print('Missing keys:', len(missing))
    print('Unexpected keys:', len(unexpected))
    if len(missing) > 0:
        print('Example missing keys:', missing[:5])
    if len(unexpected) > 0:
        print('Example unexpected keys:', unexpected[:5])

    return net

net = load_finetuned_net(MODEL_VARIANT, CHECKPOINT_PATH, device=device, dtype=dtype)

In [ ]:
# ===== 4) 初始化推理工具 =====
# 注意：首次运行会自动下载权重（若本地不存在）
BASE_CFG.download_if_needed()

rng = torch.Generator(device=device)
rng.manual_seed(SEED)

fm = FlowMatching(min_sigma=0.0, inference_mode='euler', num_steps=NUM_STEPS)

feature_utils = FeaturesUtils(
    tod_vae_ckpt=BASE_CFG.vae_path,
    synchformer_ckpt=BASE_CFG.synchformer_ckpt,
    enable_conditions=True,
    mode=BASE_CFG.mode,
    bigvgan_vocoder_ckpt=BASE_CFG.bigvgan_16k_path,
    need_vae_encoder=False,
).to(device=device, dtype=dtype).eval()

# duration 会影响序列长度
seq_cfg = BASE_CFG.seq_cfg
seq_cfg.duration = DURATION_SEC
net.update_seq_lengths(seq_cfg.latent_seq_len, seq_cfg.clip_seq_len, seq_cfg.sync_seq_len)

print('Sampling rate:', seq_cfg.sampling_rate)
print('latent_seq_len:', seq_cfg.latent_seq_len)

In [ ]:
# ===== 5) 定性评估：视频->音频生成 =====
from IPython.display import Audio, display

video_df = pd.read_csv(VIDEO_TSV, sep='\t')
print('Total candidate videos:', len(video_df))

results = []

for _, row in tqdm(video_df.head(NUM_QUALITATIVE).iterrows(), total=min(NUM_QUALITATIVE, len(video_df))):
    vid = str(row['id'])
    prompt = str(row.get('label', 'audio from video'))

    # 常见扩展名兼容
    candidates = [
        VIDEO_ROOT / f'{vid}.mp4',
        VIDEO_ROOT / f'{vid}.mov',
        VIDEO_ROOT / f'{vid}.mkv',
        VIDEO_ROOT / f'{vid}.avi',
        VIDEO_ROOT / f'{vid}.webm',
    ]
    video_path = next((p for p in candidates if p.exists()), None)
    if video_path is None:
        print(f'[Skip] video not found for id={vid}')
        continue

    try:
        video_info = load_video(video_path, duration_sec=DURATION_SEC)
        clip_frames = video_info.clip_frames.unsqueeze(0)
        sync_frames = video_info.sync_frames.unsqueeze(0)

        audios = generate(
            clip_frames,
            sync_frames,
            [prompt],
            feature_utils=feature_utils,
            net=net,
            fm=fm,
            rng=rng,
            cfg_strength=CFG_STRENGTH,
        )

        audio = audios.float().cpu()[0]
        out_path = AUDIO_OUT_DIR / f'{vid}_ft.flac'
        torchaudio.save(out_path, audio, seq_cfg.sampling_rate)

        results.append({
            'id': vid,
            'prompt': prompt,
            'video_path': str(video_path),
            'audio_path': str(out_path),
            'duration_sec': float(video_info.duration_sec),
        })

    except Exception as e:
        print(f'[Error] {vid}: {e}')

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / 'qualitative_results.csv', index=False)
print(results_df.head())
print('Saved:', OUTPUT_DIR / 'qualitative_results.csv')

# 播放第一条样本
if len(results_df) > 0:
    first_audio = results_df.iloc[0]['audio_path']
    print('试听:', first_audio)
    display(Audio(first_audio))

In [ ]:
# ===== 6) 定量评估：memmap latent 上的 flow loss =====
from torch.utils.data import DataLoader, Dataset

class MemmapEvalDataset(Dataset):
    def __init__(self, memmap_dir: Path, tsv_file: Path):
        self.tensor_dict = td.TensorDict.load_memmap(memmap_dir)
        self.meta = pd.read_csv(tsv_file, sep='\t') if tsv_file.exists() else None

        required = ['mean', 'std', 'clip_features', 'sync_features', 'text_features']
        for k in required:
            if k not in self.tensor_dict.keys():
                raise ValueError(f'Missing key in memmap: {k}')

    def __len__(self):
        return len(self.tensor_dict['mean'])

    def __getitem__(self, idx):
        return {
            'mean': self.tensor_dict['mean'][idx].clone(),
            'std': self.tensor_dict['std'][idx].clone(),
            'clip_features': self.tensor_dict['clip_features'][idx].clone(),
            'sync_features': self.tensor_dict['sync_features'][idx].clone(),
            'text_features': self.tensor_dict['text_features'][idx].clone(),
        }

@torch.no_grad()
def eval_flow_loss(net, loader, device, max_batches=20):
    net.eval()
    fm_eval = FlowMatching(min_sigma=0.0, num_steps=25)

    total_loss = 0.0
    total_n = 0

    for i, batch in enumerate(tqdm(loader, total=min(max_batches, len(loader)))):
        if i >= max_batches:
            break

        mean = batch['mean'].to(device)
        std = batch['std'].to(device)
        clip_features = batch['clip_features'].to(device)
        sync_features = batch['sync_features'].to(device)
        text_features = batch['text_features'].to(device)

        bs = mean.shape[0]
        z1 = mean + std * torch.randn_like(mean)
        z1 = net.normalize(z1)

        t = torch.rand(bs, device=device)
        z0 = torch.randn_like(z1)
        zt = fm_eval.get_conditional_flow(z0, z1, t)

        pred = net(zt, clip_features, sync_features, text_features, t)
        loss = fm_eval.loss(pred, z0, z1).mean()

        total_loss += loss.item() * bs
        total_n += bs

    return total_loss / max(total_n, 1)

if MEMMAP_DIR.exists():
    eval_ds = MemmapEvalDataset(MEMMAP_DIR, MEMMAP_TSV)
    eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    val_loss = eval_flow_loss(net, eval_loader, device=device, max_batches=NUM_QUANT_BATCHES)
    print(f'Flow-matching eval loss: {val_loss:.6f}')

    metrics = {
        'model_variant': MODEL_VARIANT,
        'checkpoint_path': str(CHECKPOINT_PATH),
        'device': str(device),
        'dtype': str(dtype),
        'num_quant_batches': NUM_QUANT_BATCHES,
        'batch_size': BATCH_SIZE,
        'flow_matching_eval_loss': float(val_loss),
    }
    with open(OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print('Saved metrics:', OUTPUT_DIR / 'metrics.json')
else:
    print(f'Memmap directory not found: {MEMMAP_DIR}')
    print('请先运行特征提取脚本，再执行本 cell。')

In [ ]:
# ===== 7) 可选：与原始预训练模型做同条件对比 =====
# 用同一批样本分别生成 base / finetuned，再做主观听感 A/B。
# 如需启用，把下面变量改成 True 并运行本单元。
RUN_BASELINE_AB = False

if RUN_BASELINE_AB:
    base_net = get_my_mmaudio(MODEL_VARIANT).to(device=device, dtype=dtype).eval()
    base_net.load_weights(torch.load(BASE_CFG.model_path, map_location=device, weights_only=True))
    base_net.update_seq_lengths(seq_cfg.latent_seq_len, seq_cfg.clip_seq_len, seq_cfg.sync_seq_len)

    ab_dir = OUTPUT_DIR / 'ab_test'
    ab_dir.mkdir(exist_ok=True)

    sample_rows = pd.read_csv(VIDEO_TSV, sep='\t').head(min(3, NUM_QUALITATIVE))
    for _, row in sample_rows.iterrows():
        vid = str(row['id'])
        prompt = str(row.get('label', 'audio from video'))
        video_path = VIDEO_ROOT / f'{vid}.mp4'
        if not video_path.exists():
            continue

        vi = load_video(video_path, duration_sec=DURATION_SEC)
        clip_frames = vi.clip_frames.unsqueeze(0)
        sync_frames = vi.sync_frames.unsqueeze(0)

        audio_base = generate(clip_frames, sync_frames, [prompt],
                              feature_utils=feature_utils, net=base_net, fm=fm, rng=rng, cfg_strength=CFG_STRENGTH).float().cpu()[0]
        audio_ft = generate(clip_frames, sync_frames, [prompt],
                            feature_utils=feature_utils, net=net, fm=fm, rng=rng, cfg_strength=CFG_STRENGTH).float().cpu()[0]

        torchaudio.save(ab_dir / f'{vid}_base.flac', audio_base, seq_cfg.sampling_rate)
        torchaudio.save(ab_dir / f'{vid}_finetuned.flac', audio_ft, seq_cfg.sampling_rate)

    print('A/B 音频已输出到:', ab_dir)
else:
    print('Skip baseline A/B (RUN_BASELINE_AB=False)')

## 使用建议（M1 Pro）

- 初次评估先把 `NUM_QUALITATIVE=3`、`NUM_QUANT_BATCHES=5`，确认流程正确后再扩大。
- 若 MPS 出现不稳定，可将 `device` 改为 `cpu` 排查。
- 如果你保存的是 `model_best.pth`，把 `CHECKPOINT_PATH` 改为该文件即可。
- 若要严格可复现，固定：`SEED`、输入视频列表、`NUM_STEPS`、`CFG_STRENGTH`。